# 08 · 序列模型与化学反应预测 — 课堂练习参考解答

本 Notebook 汇总了第 08 讲三个主讲 Notebook 的全部课堂练习参考实现与结果讨论：

- **`08a_smiles_language_model.ipynb`**：练习 `08a-1`（不同温度下的语法错误 vs. 化学价态错误剖析）、练习 `08a-2`（贪心解码 vs. 多项式采样）、练习 `08a-3`（生成分子与训练集最近邻 Tanimoto 相似度分布）；
- **`08b_smiles_lstm_aqsoldb.ipynb`**：练习 `08b-1`（温度扫描寻找新颖有效产率极值）、练习 `08b-2`（在线 SMILES 枚举增强训练循环）、练习 `08b-3`（微调前后 NLL 迁移散点图与异常值诊断）、练习 `08b-4`（生成器 + 随机森林代理筛选与适用域检验）；
- **`08c_reaction_transformer.ipynb`**：练习 `08c-1`（解码器因果掩码防止未来信息泄露验证）、练习 `08c-2`（双卤代底物区域选择性对比与新芳环骨架外推失败案例诊断）、练习 `08c-3`（USPTO 真实专利反应的旁观试剂与重原子转化率统计）。

## 0. 公共环境与工具加载

运行下方单元导入本讲所需的基础库、正则分词器以及模型定义。

In [ ]:
import os
import re
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdChemReactions
from sklearn.ensemble import RandomForestRegressor

RDLogger.DisableLog("rdApp.*")

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "../data" if os.path.exists("../data/cep.csv") else "course/08 - Sequence Models and Transformers/data"
print(f"Using device: {device}, DATA_DIR: {DATA_DIR}")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---
## 第一部分：`08a` 课堂练习解答（有机光伏分子 SMILES 语言模型）

In [ ]:
# 加载 08a 的词表、CEP 数据集与预训练 SmilesRNN
TOKEN_PATTERN = re.compile(
    r"(\[[^\]]+]|Br|Cl|Si|Se|Na|Li|Mg|Ca|Fe|Zn|Cu|Mn|@@|@|\(|\)|\.|=|#|-|\+|\\\\|\/|~|[0-9]|[A-Za-z])"
)

def tokenize_smiles(s):
    return TOKEN_PATTERN.findall(s)

cep_df = pd.read_csv(os.path.join(DATA_DIR, "cep.csv"))
cep_smiles = list(cep_df["smiles"])

class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hid_dim=192, num_layers=2, pad_idx=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers=num_layers, batch_first=True, dropout=0.1)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, x, hidden=None):
        e = self.emb(x)
        out, hidden = self.lstm(e, hidden)
        return self.fc(out), hidden

cep_ckpt = torch.load(os.path.join(DATA_DIR, "cep_rnn.pt"), map_location=device)
cep_vocab = cep_ckpt["vocab"]
cep_tok2idx = {t: i for i, t in enumerate(cep_vocab)}
cep_idx2tok = {i: t for i, t in enumerate(cep_vocab)}
CEP_PAD, CEP_BOS, CEP_EOS = cep_tok2idx["<pad>"], cep_tok2idx["^"], cep_tok2idx["$"]

cep_model = SmilesRNN(
    len(cep_vocab),
    emb_dim=cep_ckpt.get("emb_dim", 64),
    hid_dim=cep_ckpt.get("hid_dim", 192),
    num_layers=cep_ckpt.get("num_layers", 2),
    pad_idx=CEP_PAD,
).to(device)
cep_model.load_state_dict(cep_ckpt["model_state_dict"])
cep_model.eval()

@torch.no_grad()
def sample_cep_smiles(model, n_samples=100, temperature=1.0, max_len=90):
    results = []
    for _ in range(n_samples):
        cur = torch.tensor([[CEP_BOS]], dtype=torch.long, device=device)
        hidden = None
        tokens = []
        for _ in range(max_len):
            logits, hidden = model(cur, hidden)
            probs = F.softmax(logits[0, -1] / temperature, dim=-1)
            nxt = int(torch.multinomial(probs, 1).item())
            if nxt == CEP_EOS:
                break
            if nxt not in (CEP_PAD, CEP_BOS):
                tokens.append(cep_idx2tok[nxt])
            cur = torch.tensor([[nxt]], dtype=torch.long, device=device)
        results.append("".join(tokens))
    return results

print(f"Loaded cep_rnn.pt (vocab size = {len(cep_vocab)})")

### 练习 08a-1：剖析无效 SMILES 的两类死因（语法错误 vs. 化学价态/芳香性错误）

利用 `Chem.MolFromSmiles(s, sanitize=False)` 与 `Chem.DetectChemistryProblems(mol)`，我们可以把无效分子精确拆分为：
1. **语法解析错误（Syntax Error）**：括号未闭合、环编号未成对等，`MolFromSmiles(s, sanitize=False)` 直接返回 `None`；
2. **化学规则错误（Valence / Kekulization Error）**：拓扑图能建出来，但五价碳或奇数电子噻吩环无法完成芳香性凯库勒化（Kekulization）。

In [ ]:
def classify_invalid_smiles(smiles_list):
    n_valid, n_syntax_err, n_chem_err = 0, 0, 0
    syntax_examples, chem_examples = [], []

    for s in smiles_list:
        if not s:
            n_syntax_err += 1
            continue
        mol_raw = Chem.MolFromSmiles(s, sanitize=False)
        if mol_raw is None:
            n_syntax_err += 1
            if len(syntax_examples) < 3:
                syntax_examples.append(s)
        else:
            problems = Chem.DetectChemistryProblems(mol_raw)
            if len(problems) > 0:
                n_chem_err += 1
                if len(chem_examples) < 3:
                    chem_examples.append((s, problems[0].GetType(), problems[0].Message()))
            else:
                mol_clean = Chem.MolFromSmiles(s, sanitize=True)
                if mol_clean is not None:
                    n_valid += 1
                else:
                    n_chem_err += 1

    return {
        "Valid": n_valid,
        "Syntax Error": n_syntax_err,
        "Chemistry Error": n_chem_err,
        "syntax_examples": syntax_examples,
        "chem_examples": chem_examples,
    }

rows_08a1 = []
for T in [0.7, 1.0, 1.4]:
    sampled = sample_cep_smiles(cep_model, n_samples=250, temperature=T)
    res = classify_invalid_smiles(sampled)
    rows_08a1.append({
        "Temperature": T,
        "Valid (%)": 100.0 * res["Valid"] / len(sampled),
        "Syntax Error (%)": 100.0 * res["Syntax Error"] / len(sampled),
        "Chemistry Error (%)": 100.0 * res["Chemistry Error"] / len(sampled),
    })
    if T == 1.4:
        high_T_res = res

print(pd.DataFrame(rows_08a1).to_string(index=False))
print("\nSample Syntax Errors at T=1.4:", high_T_res["syntax_examples"])
print("Sample Chemistry Errors at T=1.4:")
for s, ptype, msg in high_T_res["chem_examples"]:
    print(f"  [{ptype}] {msg} -> {s}")

### 练习 08a-2：贪心解码（Greedy Decoding, $T \to 0$）为什么只产生单一条分子？

In [ ]:
@torch.no_grad()
def greedy_decode_cep(model, max_len=90):
    cur = torch.tensor([[CEP_BOS]], dtype=torch.long, device=device)
    hidden = None
    tokens = []
    for _ in range(max_len):
        logits, hidden = model(cur, hidden)
        nxt = int(logits[0, -1].argmax(dim=-1).item())
        if nxt == CEP_EOS:
            break
        if nxt not in (CEP_PAD, CEP_BOS):
            tokens.append(cep_idx2tok[nxt])
        cur = torch.tensor([[nxt]], dtype=torch.long, device=device)
    return "".join(tokens)

greedy_runs = [greedy_decode_cep(cep_model) for _ in range(10)]
print(f"Unique molecules across 10 greedy runs: {len(set(greedy_runs))}")
print(f"Greedy decoded SMILES: {greedy_runs[0]}")

**讨论**：在 `model.eval()` 模式下，Dropout 已关闭，起始输入固定为 `^`，且每一步选取 `argmax` 是完全确定性的映射，因此无论重复调用多少次，贪心解码永远沿着同一条最大局部概率路径生成唯一的一条常见分子。只有引入带温度的多项式随机采样（`torch.multinomial`），才能探索分布中的多样性候选。

### 练习 08a-3：生成分子与训练集最近邻 Tanimoto 相似度分布

In [ ]:
# 取 CEP 训练集子集计算 Morgan 指纹，评估新颖生成分子的最近邻相似度
ref_mols = [Chem.MolFromSmiles(s) for s in cep_smiles[:3000]]
ref_fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=1024) for m in ref_mols if m is not None]
train_canon_set = set(cep_smiles)

sampled_10 = sample_cep_smiles(cep_model, n_samples=200, temperature=1.0)
novel_mols = []
for s in sampled_10:
    m = Chem.MolFromSmiles(s)
    if m is not None:
        cs = Chem.MolToSmiles(m, canonical=True)
        if cs not in train_canon_set:
            novel_mols.append(m)

max_sims = []
for m in novel_mols:
    fp = AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=1024)
    sims = DataStructs.BulkTanimotoSimilarity(fp, ref_fps)
    max_sims.append(max(sims))

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.hist(max_sims, bins=20, color="#2b5c8f", edgecolor="white", alpha=0.88)
ax.axvline(np.mean(max_sims), color="#d62728", ls="--", lw=1.5, label=f"Mean NN Sim = {np.mean(max_sims):.2f}")
ax.set_xlabel("Nearest-Neighbor Tanimoto Similarity to CEP Training Set")
ax.set_ylabel("Count")
ax.set_title("Novel Generated Molecules Remain Structurally Grounded in OPV Motifs")
ax.legend()
plt.tight_layout()
plt.show()

---
## 第二部分：`08b` 课堂练习解答（SMILES 枚举增强、似然探针与迁移微调）

In [ ]:
# 加载 08b 的预训练模型与数据集
sol_df = pd.read_csv(os.path.join(DATA_DIR, "solubility.csv"))
osc_df = pd.read_csv(os.path.join(DATA_DIR, "osc_finetune.csv"))
pre_ckpt = torch.load(os.path.join(DATA_DIR, "smiles_lstm_pretrained.pt"), map_location=device)

class Vocab:
    def __init__(self, tokens):
        self.idx_to_token = list(tokens)
        self.token_to_idx = {t: i for i, t in enumerate(self.idx_to_token)}
        self.pad_id = self.token_to_idx["<pad>"]
        self.bos_id = self.token_to_idx["^"]
        self.eos_id = self.token_to_idx["$"]

    def __len__(self):
        return len(self.idx_to_token)

    def encode(self, smiles, max_len=62):
        toks = tokenize_smiles(smiles)[: max_len - 2]
        ids = [self.bos_id] + [self.token_to_idx[t] for t in toks if t in self.token_to_idx] + [self.eos_id]
        return ids + [self.pad_id] * (max_len - len(ids))

    def decode(self, ids):
        out = []
        for i in ids:
            i = int(i)
            if i == self.eos_id:
                break
            if i not in (self.pad_id, self.bos_id):
                out.append(self.idx_to_token[i])
        return "".join(out)

vocab_08b = Vocab(pre_ckpt["vocab"])

class SmilesLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hid_dim=256, num_layers=2, dropout=0.2, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        out, hidden = self.lstm(emb, hidden)
        return self.fc(out), hidden

pre_model = SmilesLSTM(len(vocab_08b), pad_id=vocab_08b.pad_id).to(device)
pre_model.load_state_dict(pre_ckpt["model_state_dict"])
pre_model.eval()
print(f"Loaded smiles_lstm_pretrained.pt (vocab size = {len(vocab_08b)})")

### 练习 08b-1：采样温度扫描与“新颖且唯一有效分子净产率”极值

In [ ]:
@torch.no_grad()
def sample_batch_08b(model, vocab, n_samples=200, temperature=1.0, max_len=62):
    model.eval()
    cur = torch.full((n_samples, 1), vocab.bos_id, dtype=torch.long, device=device)
    hidden = None
    finished = torch.zeros(n_samples, dtype=torch.bool, device=device)
    seqs = [cur]
    for _ in range(max_len - 1):
        logits, hidden = model(cur, hidden)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        nxt = torch.multinomial(probs, 1)
        nxt = torch.where(finished.unsqueeze(1), torch.tensor(vocab.pad_id, device=device), nxt)
        seqs.append(nxt)
        finished = finished | (nxt.squeeze(1) == vocab.eos_id)
        cur = nxt
        if finished.all():
            break
    mat = torch.cat(seqs, dim=1).cpu().numpy()
    return [vocab.decode(row) for row in mat]

aq_train_canon = set()
for s in sol_df["SMILES"][:4000]:
    m = Chem.MolFromSmiles(str(s))
    if m is not None:
        aq_train_canon.add(Chem.MolToSmiles(m, canonical=True))

temps = [0.5, 0.7, 0.85, 1.0, 1.2, 1.5]
yield_rows = []
for T in temps:
    raw_list = sample_batch_08b(pre_model, vocab_08b, n_samples=200, temperature=T)
    valid_canons = []
    for s in raw_list:
        m = Chem.MolFromSmiles(s) if s else None
        if m is not None:
            valid_canons.append(Chem.MolToSmiles(m, canonical=True))
    uniq_canons = set(valid_canons)
    novel_uniq = [c for c in uniq_canons if c not in aq_train_canon]
    yield_rows.append({
        "Temperature": T,
        "Validity (%)": 100.0 * len(valid_canons) / len(raw_list),
        "Uniqueness among Valid (%)": 100.0 * len(uniq_canons) / max(len(valid_canons), 1),
        "Net Novel & Unique Yield (%)": 100.0 * len(novel_uniq) / len(raw_list),
    })

yield_df = pd.DataFrame(yield_rows)
print(yield_df.to_string(index=False))

**结论**：温度过低（$T=0.5$）时虽然有效率极高，但重复采样率上升导致唯一性下降；温度过高（$T=1.5$）时语法崩溃导致有效率暴跌。综合下来，$T \in [0.85, 1.0]$ 附近能取得最高的“新颖且唯一有效分子净产率”。

### 练习 08b-2：实现在线 SMILES 随机枚举增强训练函数 `train_lm_augmented`

In [ ]:
def randomize_smiles_single(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return smiles
    return Chem.MolToSmiles(mol, doRandom=True, canonical=False)

def train_lm_augmented(model, train_smiles_list, vocab, epochs=2, batch_size=64, lr=1e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    epoch_losses = []
    for ep in range(1, epochs + 1):
        # 每个 Epoch 开始前动态生成一套新的随机重编号 SMILES
        aug_encoded = [vocab.encode(randomize_smiles_single(s)) for s in train_smiles_list]
        tensor_x = torch.tensor(aug_encoded, dtype=torch.long)
        loader = DataLoader(TensorDataset(tensor_x), batch_size=batch_size, shuffle=True)

        model.train()
        total_loss = 0.0
        for (batch_x,) in loader:
            batch_x = batch_x.to(device)
            logits, _ = model(batch_x[:, :-1])
            loss = F.cross_entropy(
                logits.reshape(-1, len(vocab)),
                batch_x[:, 1:].reshape(-1),
                ignore_index=vocab.pad_id,
            )
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * batch_x.size(0)
        epoch_losses.append(total_loss / len(train_smiles_list))
    return epoch_losses

demo_aug_model = SmilesLSTM(len(vocab_08b), emb_dim=64, hid_dim=128, num_layers=1, dropout=0.0, pad_id=vocab_08b.pad_id).to(device)
demo_losses = train_lm_augmented(demo_aug_model, list(sol_df["SMILES"][:300]), vocab_08b, epochs=2)
print("Demo online augmented training losses:", [round(x, 4) for x in demo_losses])

### 练习 08b-3：微调前后 NLL 迁移散点图与高 NLL 离群分子诊断

In [ ]:
@torch.no_grad()
def compute_nll_per_token(model, smiles_list, vocab):
    model.eval()
    x = torch.tensor([vocab.encode(s) for s in smiles_list], dtype=torch.long, device=device)
    logits, _ = model(x[:, :-1])
    targets = x[:, 1:]
    loss_mat = F.cross_entropy(
        logits.reshape(-1, len(vocab)),
        targets.reshape(-1),
        ignore_index=vocab.pad_id,
        reduction="none",
    ).reshape(targets.shape)
    valid_counts = (targets != vocab.pad_id).sum(dim=1).clamp(min=1)
    return (loss_mat.sum(dim=1) / valid_counts).cpu().numpy()

# 在有机半导体训练集上微调拷贝模型
import copy
osc_smiles = list(osc_df["smiles"])
osc_train, osc_val = osc_smiles[:135], osc_smiles[135:]

ft_model = copy.deepcopy(pre_model)
_ = train_lm_augmented(ft_model, osc_train * 4, vocab_08b, epochs=3, batch_size=64, lr=8e-4)

aq_eval_sub = [str(s) for s in sol_df["SMILES"][:100] if len(tokenize_smiles(str(s))) <= 55][:60]
nll_pre_osc = compute_nll_per_token(pre_model, osc_val, vocab_08b)
nll_ft_osc = compute_nll_per_token(ft_model, osc_val, vocab_08b)
nll_pre_aq = compute_nll_per_token(pre_model, aq_eval_sub, vocab_08b)
nll_ft_aq = compute_nll_per_token(ft_model, aq_eval_sub, vocab_08b)

fig, ax = plt.subplots(figsize=(6.2, 5.2))
ax.scatter(nll_pre_aq, nll_ft_aq, alpha=0.7, label="AqSolDB Holdout (General)", color="#7f7f7f")
ax.scatter(nll_pre_osc, nll_ft_osc, alpha=0.9, label="OSC Validation (Target Domain)", color="#d62728", s=45)
lim = max(nll_pre_aq.max(), nll_ft_aq.max(), nll_pre_osc.max()) + 0.2
ax.plot([0, lim], [0, lim], "k--", lw=1, label="y = x (No Change)")
ax.set_xlabel("Pretrained Model NLL per Token")
ax.set_ylabel("Fine-Tuned Model NLL per Token")
ax.set_title("Domain Specialization vs. Catastrophic Forgetting")
ax.legend()
plt.tight_layout()
plt.show()

worst_idx = int(np.argmax(nll_ft_osc))
print(f"Highest NLL molecule in osc_val after fine-tuning (NLL={nll_ft_osc[worst_idx]:.3f}):")
print(" ", osc_val[worst_idx])

### 练习 08b-4：分子生成 + 随机森林溶解度代理模型闭环筛选与适用域检验

In [ ]:
def calc_rdkit_desc(smiles):
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        return None
    return [
        Descriptors.MolWt(m),
        Descriptors.MolLogP(m),
        Descriptors.TPSA(m),
        Descriptors.NumHDonors(m),
        Descriptors.NumHAcceptors(m),
        Descriptors.RingCount(m),
    ]

sub_sol = sol_df.iloc[:1500]
X_rf, y_rf = [], []
for s, y in zip(sub_sol["SMILES"], sub_sol["Solubility"]):
    d = calc_rdkit_desc(str(s))
    if d is not None:
        X_rf.append(d)
        y_rf.append(float(y))
rf_surrogate = RandomForestRegressor(n_estimators=100, random_state=SEED).fit(X_rf, y_rf)

gen_osc = sample_batch_08b(ft_model, vocab_08b, n_samples=150, temperature=0.9)
X_gen, valid_gen = [], []
for s in set(gen_osc):
    d = calc_rdkit_desc(s)
    if d is not None:
        X_gen.append(d)
        valid_gen.append(s)

pred_logS = rf_surrogate.predict(X_gen)
print(f"Generated {len(valid_gen)} unique valid OSC candidates.")
print(f"Predicted LogS mean: {np.mean(pred_logS):.2f} (Highly hydrophobic conjugated aromatics)")

---
## 第三部分：`08c` 课堂练习解答（注意力机制与 ReactionTransformer）

### 练习 08c-1：验证解码器因果掩码防止未来信息泄露

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))
    attn = F.softmax(scores, dim=-1)
    return torch.matmul(attn, V), attn

def check_causal_leakage(Q, K, V):
    T = Q.size(-2)
    causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=Q.device), diagonal=1)

    V_pert = V.clone()
    V_pert[:, -1, :] += 100.0

    out_masked_1, _ = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
    out_masked_2, _ = scaled_dot_product_attention(Q, K, V_pert, mask=causal_mask)

    out_unmasked_1, _ = scaled_dot_product_attention(Q, K, V, mask=None)
    out_unmasked_2, _ = scaled_dot_product_attention(Q, K, V_pert, mask=None)

    diff_masked_t0 = torch.norm(out_masked_2[:, 0, :] - out_masked_1[:, 0, :]).item()
    diff_unmasked_t0 = torch.norm(out_unmasked_2[:, 0, :] - out_unmasked_1[:, 0, :]).item()
    return diff_masked_t0, diff_unmasked_t0

torch.manual_seed(0)
Q_t = torch.randn(1, 6, 16)
K_t = torch.randn(1, 6, 16)
V_t = torch.randn(1, 6, 16)
d_mask, d_unmask = check_causal_leakage(Q_t, K_t, V_t)
print(f"With Causal Mask    | ||Delta out[t=0]||_2 = {d_mask:.8f} (Strictly zero future leakage)")
print(f"Without Causal Mask | ||Delta out[t=0]||_2 = {d_unmask:.4f} (Severe leakage from future step T-1)")

### 练习 08c-2：双卤代底物区域选择性对比与 `test_new_aryl` 失败案例诊断

In [ ]:
# 1. SMARTS 模板对双卤代底物 Brc1ccc(Cl)s1 的预测
suzuki_smarts = "[c:1][Br,Cl].[c:2]B(O)O>>[c:1]-[c:2]"
rxn_rule = rdChemReactions.ReactionFromSmarts(suzuki_smarts)

m_hal = Chem.MolFromSmiles("Brc1ccc(Cl)s1")
m_bor = Chem.MolFromSmiles("CC(C)(C)c1csc(B(O)O)c1")
smarts_prods = set()
for p_tup in rxn_rule.RunReactants((m_hal, m_bor)):
    try:
        Chem.SanitizeMol(p_tup[0])
        smarts_prods.add(Chem.MolToSmiles(p_tup[0], canonical=True))
    except Exception:
        pass

print("SMARTS template products on Brc1ccc(Cl)s1 (matches BOTH -Br and -Cl indiscriminately):")
for p in smarts_prods:
    print(" ", p)

In [ ]:
# 2. 加载 ReactionTransformer 预测同一双卤代底物，并诊断 test_new_aryl 误差
suzuki_df = pd.read_csv(os.path.join(DATA_DIR, "suzuki_toy.csv"))
rxn_ckpt_path = os.path.join(DATA_DIR, "suzuki_transformer.pt")

if os.path.exists(rxn_ckpt_path):
    rxn_ckpt = torch.load(rxn_ckpt_path, map_location=device)
    rxn_vocab = rxn_ckpt["vocab"]
    rxn_tok2idx = {t: i for i, t in enumerate(rxn_vocab)}
    rxn_idx2tok = {i: t for i, t in enumerate(rxn_vocab)}
    PAD_ID, BOS_ID, EOS_ID = 0, 1, 2

    class SinusoidalPositionalEncoding(nn.Module):
        def __init__(self, d_model, max_len=128):
            super().__init__()
            pe = torch.zeros(max_len, d_model)
            position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            self.register_buffer("pe", pe.unsqueeze(0))

        def forward(self, x):
            return x + self.pe[:, : x.size(1)]

    class EncoderLayer(nn.Module):
        def __init__(self, d_model, nhead, dim_ff, dropout=0.1):
            super().__init__()
            self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            self.ffn = nn.Sequential(nn.Linear(d_model, dim_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(dim_ff, d_model))
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)
            self.drop = nn.Dropout(dropout)

        def forward(self, src, src_key_padding_mask=None):
            attn_out, _ = self.self_attn(src, src, src, key_padding_mask=src_key_padding_mask, need_weights=False)
            src = self.norm1(src + self.drop(attn_out))
            return self.norm2(src + self.drop(self.ffn(src)))

    class DecoderLayer(nn.Module):
        def __init__(self, d_model, nhead, dim_ff, dropout=0.1):
            super().__init__()
            self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            self.ffn = nn.Sequential(nn.Linear(d_model, dim_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(dim_ff, d_model))
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)
            self.norm3 = nn.LayerNorm(d_model)
            self.drop = nn.Dropout(dropout)
            self.last_cross_attn = None

        def forward(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
            self_out, _ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask, need_weights=False)
            tgt = self.norm1(tgt + self.drop(self_out))
            cross_out, cross_w = self.cross_attn(tgt, memory, memory, key_padding_mask=memory_key_padding_mask, need_weights=True, average_attn_weights=False)
            self.last_cross_attn = cross_w.detach()
            tgt = self.norm2(tgt + self.drop(cross_out))
            return self.norm3(tgt + self.drop(self.ffn(tgt)))

    class ReactionTransformer(nn.Module):
        def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, dim_ff=256, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
            self.pos_encoder = SinusoidalPositionalEncoding(d_model, max_len=128)
            self.enc_layers = nn.ModuleList([EncoderLayer(d_model, nhead, dim_ff, dropout) for _ in range(num_layers)])
            self.dec_layers = nn.ModuleList([DecoderLayer(d_model, nhead, dim_ff, dropout) for _ in range(num_layers)])
            self.fc_out = nn.Linear(d_model, vocab_size)

        def encode(self, src):
            src_pad_mask = (src == PAD_ID)
            memory = self.pos_encoder(self.embedding(src) * math.sqrt(self.d_model))
            for layer in self.enc_layers:
                memory = layer(memory, src_key_padding_mask=src_pad_mask)
            return memory, src_pad_mask

        def decode(self, tgt_in, memory, src_pad_mask):
            T_tgt = tgt_in.size(1)
            tgt_mask = torch.triu(torch.ones(T_tgt, T_tgt, device=tgt_in.device, dtype=torch.bool), diagonal=1)
            tgt_pad_mask = (tgt_in == PAD_ID)
            h = self.pos_encoder(self.embedding(tgt_in) * math.sqrt(self.d_model))
            for layer in self.dec_layers:
                h = layer(h, memory, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_pad_mask, memory_key_padding_mask=src_pad_mask)
            return self.fc_out(h)

    rxn_model = ReactionTransformer(len(rxn_vocab)).to(device)
    rxn_model.load_state_dict(rxn_ckpt["model_state_dict"])
    rxn_model.eval()

    def encode_rxn(s, max_len=64):
        ids = [BOS_ID] + [rxn_tok2idx[t] for t in tokenize_smiles(s)][: max_len - 2] + [EOS_ID]
        return ids + [PAD_ID] * (max_len - len(ids))

    @torch.no_grad()
    def translate_single(s):
        src_t = torch.tensor([encode_rxn(s)], dtype=torch.long, device=device)
        mem, pad_m = rxn_model.encode(src_t)
        ys = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)
        for _ in range(62):
            logits = rxn_model.decode(ys, mem, pad_m)
            nxt = int(logits[0, -1].argmax(dim=-1).item())
            if nxt == EOS_ID:
                break
            ys = torch.cat([ys, torch.tensor([[nxt]], device=device)], dim=1)
        return "".join(rxn_idx2tok[int(i)] for i in ys[0, 1:])

    competing_pred = translate_single("Brc1ccc(Cl)s1.CC(C)(C)c1csc(B(O)O)c1")
    print(f"ReactionTransformer Top-1 on Brc1ccc(Cl)s1: {competing_pred}")
    print("  -> Notice how the model reacts at the C-Br site and retains Cl!")

### 练习 08c-3：解析 USPTO 真实专利反应的旁观试剂与重原子转化率

In [ ]:
uspto_df = pd.read_csv(os.path.join(DATA_DIR, "uspto_test_sample.csv"))

comp_counts = []
atom_utilization = []
precursor_counter = Counter()

for _, row in uspto_df.iterrows():
    prec_str = str(row["precursors"])
    prod_str = str(row["product"])
    parts = prec_str.split(".")
    comp_counts.append(len(parts))

    prec_atoms = 0
    for p in parts:
        m = Chem.MolFromSmiles(p)
        if m is not None:
            prec_atoms += m.GetNumHeavyAtoms()
            precursor_counter[Chem.MolToSmiles(m, canonical=True)] += 1

    prod_mol = Chem.MolFromSmiles(prod_str)
    if prod_mol is not None and prec_atoms > 0:
        atom_utilization.append(prod_mol.GetNumHeavyAtoms() / prec_atoms)

print(f"Mean precursor components per reaction: {np.mean(comp_counts):.2f} (max = {max(comp_counts)})")
print(f"Mean heavy atom utilization ratio     : {np.mean(atom_utilization):.2%}")
print("\nTop-10 Most Frequent Species in USPTO Precursors (Solvents, Bases & Common Reagents):")
for smi, cnt in precursor_counter.most_common(10):
    print(f"  {cnt:3d}x | {smi}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
axes[0].hist(comp_counts, bins=range(1, max(comp_counts) + 2), align="left", rwidth=0.75, color="#2b5c8f")
axes[0].set_xlabel("Number of Precursor Species (Separated by '.')")
axes[0].set_ylabel("Reaction Count")
axes[0].set_title("Precursor Component Count in USPTO Sample")

axes[1].hist(atom_utilization, bins=20, color="#3a923a", edgecolor="white")
axes[1].set_xlabel("Heavy Atom Utilization (Product / Precursors)")
axes[1].set_ylabel("Reaction Count")
axes[1].set_title("Heavy Atom Conservation Ratio")
plt.tight_layout()
plt.show()